# Analiza koszyka zakupowego

Celem tego notebooka jest analiza produktów kupowanych razem w ramach jednego zamówienia.

Analiza koszyka zakupowego pozwala:

- identyfikować produkty często kupowane razem
- wskazywać potencjalne zależności zakupowe
- wspierać działania cross-sellingowe
- lepiej zrozumieć zachowania klientów

W projekcie analizowane będą najczęściej występujące pary produktów w zamówieniach.

## Import bibliotek

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from itertools import combinations
from collections import Counter

sns.set_style("whitegrid")

## Wczytanie danych

In [ ]:
df = pd.read_csv("../data/processed/sales_cleaned.csv")

df.head()

## Przygotowanie danych do analizy koszyka

Analiza koszyka wymaga danych na poziomie zamówienia.

Dla każdego zamówienia tworzymy listę produktów znajdujących się w koszyku.

In [ ]:
basket = (
    df.groupby("order_id")["product_id"]
    .apply(list)
    .reset_index()
)

basket.head()

## Wybór zamówień zawierających więcej niż jeden produkt

Analiza produktów kupowanych razem ma sens tylko dla zamówień, w których znajduje się więcej niż jeden produkt.

In [ ]:
basket["basket_size"] = basket["product_id"].apply(len)

multi_item_baskets = basket[basket["basket_size"] > 1].copy()

multi_item_baskets.head()

In [ ]:
print("Liczba wszystkich zamówień:", basket["order_id"].nunique())
print("Liczba zamówień wieloproduktowych:", multi_item_baskets["order_id"].nunique())

### Interpretacja

Nie wszystkie zamówienia nadają się do analizy koszyka.

Do dalszej analizy wykorzystujemy wyłącznie zamówienia zawierające więcej niż jeden produkt.

## Generowanie par produktów

Dla każdego zamówienia generujemy wszystkie unikalne pary produktów występujących razem w koszyku.

In [ ]:
pair_counter = Counter()

for products_in_order in multi_item_baskets["product_id"]:
    unique_products = sorted(set(products_in_order))
    
    for pair in combinations(unique_products, 2):
        pair_counter[pair] += 1

In [ ]:
top_pairs = (
    pd.DataFrame(pair_counter.items(), columns=["product_pair", "count"])
    .sort_values("count", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top_pairs

## Najczęściej występujące pary produktów

In [ ]:
top_pairs["product_pair"] = top_pairs["product_pair"].astype(str)

top_pairs

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=top_pairs,
    x="count",
    y="product_pair"
)

plt.title("Top 10 par produktów kupowanych razem")
plt.tight_layout()

plt.savefig("../reports/figures/top_product_pairs.png")
plt.show()

### Interpretacja

Najczęściej występujące pary produktów wskazują produkty, które klienci mają tendencję kupować razem.

Może to stanowić podstawę do:

- rekomendacji produktowych
- ofert łączonych
- działań cross-sellingowych
- optymalizacji układu oferty sklepu

## Rozkład wielkości koszyka

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    basket["basket_size"],
    bins=20
)

plt.title("Rozkład liczby produktów w zamówieniu")
plt.tight_layout()

plt.savefig("../reports/figures/basket_size_distribution.png")
plt.show()

### Interpretacja

Rozkład wielkości koszyka pokazuje, czy klienci częściej kupują pojedyncze produkty, czy składają większe zamówienia.

Ma to znaczenie dla strategii sprzedażowej oraz projektowania rekomendacji.

## Zapis wyników analizy koszyka

In [ ]:
top_pairs.to_csv("../data/processed/top_product_pairs.csv", index=False)

## Podsumowanie

Analiza koszyka zakupowego pozwoliła:

- wyodrębnić zamówienia zawierające wiele produktów
- zidentyfikować najczęściej występujące pary produktów
- wskazać potencjalne możliwości cross-sellingu

Wyniki tej analizy mogą zostać wykorzystane w dashboardzie oraz w rekomendacjach biznesowych.